# Ia 型超新星で宇宙の膨張の歴史を当てはめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/cosmology/sn_cosmology_fit.ipynb)

[distance/sn_ia_hubble.ipynb](../distance/sn_ia_hubble.ipynb) では、超新星の距離からハッブル定数 $H_0$ を出し、ハッブル図を 2 つの宇宙モデル(ΛCDM と物質だけの宇宙)と見比べて、宇宙が加速膨張していることを確かめました。

このノートブックでは、見比べるだけでなく、同じデータに **宇宙モデルのパラメータを当てはめます**。

1. 平坦な ΛCDM で、物質の割合 $\Omega_m$ を求める
2. 空間の曲率も自由にして、$\Omega_m$ とダークエネルギーの割合 $\Omega_\Lambda$ を同時に求める(1998 年の加速膨張の発見と同じ図)
3. ダークエネルギーの性質を表す状態方程式 $w$ を求める(宇宙項なら $w = -1$)
4. 求めたパラメータから、減速パラメータ $q(z)$、加速に転じた時期、膨張の速さの変化、宇宙の年齢を計算する

## 使うデータ

- **Pantheon+**(Scolnic et al. 2022, Brout et al. 2022)の距離データと、統計誤差+系統誤差の **共分散行列**:
  https://github.com/PantheonPlusSH0ES/DataRelease
  - リポジトリにライセンスの明記はありません。研究用に公開されたデータなので、使うときは上の論文を引用します。

前のノートブックでは、超新星ごとに単純に平均して誤差を対角だけで扱いました。
宇宙論パラメータを決めるときは、**較正のずれのように多くの超新星に共通してかかる系統誤差** が効いてくるので、ここでは共分散行列をそのまま使います。

## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2 as chi2_dist

BASE_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
)
DATA_URL = BASE_URL + "Pantheon%2BSH0ES.dat"
COV_URL = BASE_URL + "Pantheon%2BSH0ES_STAT%2BSYS.cov"  # 約 33 MB
C_KMS = 299_792.458  # 光速 [km/s]
HUBBLE_TIME_GYR = 977.79  # 1/H0 [Gyr] × H0 [km/s/Mpc]

## 1. データと共分散行列を読み込む

共分散行列のファイルは、1 行目が行数 $N$、そのあとに $N \times N$ 個の数字が 1 行に 1 つずつ並んでいます。
行の順番は、距離データ(`.dat`)の行の順番と同じです。

In [ ]:
raw = pd.read_csv(DATA_URL, sep=r"\s+")
cov_flat = pd.read_csv(COV_URL, header=None)[0].to_numpy()
n_all = int(cov_flat[0])
cov_all = cov_flat[1:].reshape(n_all, n_all)
print(f"{len(raw)} 行, 共分散行列 {n_all} × {n_all}")

宇宙論の当てはめには、Brout et al. (2022) と同じく

- $z > 0.01$(これより近いと、銀河の固有運動による赤方偏移の乱れが大きい)
- セファイド較正用の超新星(`IS_CALIBRATOR = 1`)は除く(これらは 6 章で $H_0$ を出すのに使う)

の行を使います。同じ超新星を複数のプロジェクトが測った重複行は、共分散行列に相関として入っているので、そのまま残します。

In [ ]:
use = ((raw["zHD"] > 0.01) & (raw["IS_CALIBRATOR"] == 0)).to_numpy()
sn = raw[use].reset_index(drop=True)
cov = cov_all[np.ix_(use, use)]
cov_inv = np.linalg.inv(cov)

m_obs = sn["m_b_corr"].to_numpy()  # 光度曲線の幅と色で補正した見かけの等級
z_hd = sn["zHD"].to_numpy()        # 宇宙背景放射の静止系での赤方偏移(固有運動も補正)
z_hel = sn["zHEL"].to_numpy()      # 太陽系から見た赤方偏移
print(f"使う行: {len(sn)}(超新星 {sn['CID'].nunique()} 個), z = {z_hd.min():.3f} 〜 {z_hd.max():.2f}")

err_diag = np.sqrt(np.diag(cov))
plt.figure(figsize=(7, 4))
plt.errorbar(z_hd, m_obs, yerr=err_diag, fmt=".", ms=2, alpha=0.3)
plt.xscale("log")
plt.xlabel("redshift zHD")
plt.ylabel("m_B (corrected) [mag]")
plt.gca().invert_yaxis()
plt.show()

## 2. 宇宙モデルから光度距離を計算する

宇宙の膨張の速さ $H(z)$ は、フリードマン方程式で物質・曲率・ダークエネルギーの割合から決まります。

$$
E(z) \equiv \frac{H(z)}{H_0} = \sqrt{\Omega_m (1+z)^3 + \Omega_k (1+z)^2 + \Omega_{\mathrm{DE}} (1+z)^{3(1+w)}},
\qquad \Omega_k = 1 - \Omega_m - \Omega_{\mathrm{DE}}
$$

($w = -1$ のときダークエネルギーは宇宙項 Λ になり、密度が時間によらず一定です。放射の寄与は $z < 2.3$ では無視できるほど小さいので入れていません。)

光が届くまでに進む共動距離は $D_C = \dfrac{c}{H_0}\displaystyle\int_0^z \frac{dz'}{E(z')}$ で、
空間が曲がっているとき($\Omega_k \neq 0$)は

$$
D_M = \frac{c}{H_0} \frac{1}{\sqrt{|\Omega_k|}}\, S\!\left(\sqrt{|\Omega_k|}\, \frac{H_0 D_C}{c}\right), \qquad
S = \sinh\ (\Omega_k > 0),\ \sin\ (\Omega_k < 0)
$$

とします。光度距離は $d_L = (1 + z_{\mathrm{hel}})\, D_M(z_{\mathrm{HD}})$ です
($1 + z$ の因子は、光子のエネルギーが下がることと届く頻度が下がることから来るもので、観測者から見た赤方偏移 $z_{\mathrm{hel}}$ を使います)。

多数のモデルを一度に計算できるよう、パラメータを配列で受け取る関数にします。
$E^2 \le 0$ になる(宇宙が過去にビッグバンを経ずに収縮から跳ね返った)モデルは、計算できないので印をつけて除きます。

In [ ]:
Z_GRID = np.linspace(0, z_hd.max() * 1.01, 3000)


def e_of_z(z, om, ode, w=-1.0):
    # E(z) = H(z)/H0 の 2 乗。om, ode, w は形 (P, 1) の配列、z は形 (1, Z) の配列
    ok = 1 - om - ode
    zp = 1 + z
    return om * zp ** 3 + ok * zp ** 2 + ode * zp ** (3 * (1 + w))


def dl_over_dh(om, ode, w=-1.0):
    # H0 d_L / c を、各モデル(P 個)× 各超新星(N 個)について返す。計算できないモデルは bad=True
    om, ode, w = (np.atleast_1d(v).astype(float)[:, None] for v in np.broadcast_arrays(om, ode, w))
    e2 = e_of_z(Z_GRID[None, :], om, ode, w)
    bad = np.any(e2 <= 0, axis=1)
    inv_e = 1 / np.sqrt(np.clip(e2, 1e-12, None))
    # 台形公式で積分して共動距離
    dc = np.concatenate(
        [np.zeros((len(om), 1)), np.cumsum((inv_e[:, 1:] + inv_e[:, :-1]) / 2 * np.diff(Z_GRID), axis=1)], axis=1
    )
    ok = 1 - om - ode
    sk = np.sqrt(np.abs(ok))
    sk_safe = np.where(sk > 0, sk, 1)
    with np.errstate(over="ignore", invalid="ignore"):
        dm = np.where(ok > 1e-8, np.sinh(sk * dc) / sk_safe, np.where(ok < -1e-8, np.sin(sk * dc) / sk_safe, dc))
    # 閉じた宇宙で sin が折り返すモデルも除く
    bad |= ~np.all(np.isfinite(dm[:, 1:]) & (dm[:, 1:] > 0), axis=1)
    dm = np.where(bad[:, None], 1.0, dm)
    out = np.array([np.interp(z_hd, Z_GRID, row) for row in dm]) * (1 + z_hel)[None, :]
    return out, bad


# 確認: 平坦 ΛCDM なら astropy と同じ値になるはず
from astropy.cosmology import FlatLambdaCDM

dl, _ = dl_over_dh(0.3, 0.7)
dl_astropy = FlatLambdaCDM(H0=70, Om0=0.3, Tcmb0=0).comoving_distance(z_hd).value * (1 + z_hel) * 70 / C_KMS
print(f"astropy との差の最大値: {np.max(np.abs(dl[0] / dl_astropy - 1)):.1e}")

### 絶対等級と $H_0$ はまとめて 1 つの定数になる

見かけの等級のモデルは

$$
m = M_B + 5 \log_{10}\frac{d_L}{10\,\mathrm{pc}}
  = 5 \log_{10}\frac{H_0 d_L}{c} + \underbrace{M_B + 5\log_{10}\frac{c/H_0}{1\,\mathrm{Mpc}} + 25}_{\mathcal{A}}
$$

です。超新星の絶対等級 $M_B$ と $H_0$ は $\mathcal{A}$ という組み合わせでしか出てこないので、超新星のハッブル図だけでは別々に決まりません。
宇宙の **形**(膨張の歴史)を決める $\Omega_m, \Omega_\Lambda, w$ は、$\mathcal{A}$ がいくつでも関係なく求まります。

そこで、各モデルで $\chi^2$ が最小になる $\mathcal{A}$ を解析的に選びます。残差を $\boldsymbol{r} = \boldsymbol{m} - 5\log_{10}(H_0 d_L/c)$、共分散行列を $C$ として

$$
\chi^2(\mathcal{A}) = (\boldsymbol{r} - \mathcal{A}\boldsymbol{1})^T C^{-1} (\boldsymbol{r} - \mathcal{A}\boldsymbol{1}), \qquad
\hat{\mathcal{A}} = \frac{\boldsymbol{1}^T C^{-1} \boldsymbol{r}}{\boldsymbol{1}^T C^{-1} \boldsymbol{1}}, \qquad
\chi^2_{\min} = \boldsymbol{r}^T C^{-1}\boldsymbol{r} - \frac{(\boldsymbol{1}^T C^{-1}\boldsymbol{r})^2}{\boldsymbol{1}^T C^{-1}\boldsymbol{1}}
$$

In [ ]:
ONE = np.ones(len(m_obs))
S11 = ONE @ cov_inv @ ONE


def chi2_models(om, ode, w=-1.0, chunk=2000):
    # 各モデルの χ²(A を最適化したもの)と、そのときの A を返す
    om, ode, w = (np.ravel(v) for v in np.broadcast_arrays(om, ode, w))
    c2_all, a_all = [], []
    for k in range(0, len(om), chunk):
        dl, bad = dl_over_dh(om[k:k + chunk], ode[k:k + chunk], w[k:k + chunk])
        r = m_obs[None, :] - 5 * np.log10(dl)
        rc = r @ cov_inv
        s_rr = np.einsum("ij,ij->i", rc, r)
        s_1r = rc @ ONE
        c2 = s_rr - s_1r ** 2 / S11
        c2[bad] = np.inf
        c2_all.append(c2)
        a_all.append(s_1r / S11)
    return np.concatenate(c2_all), np.concatenate(a_all)


def posterior_stats(values, c2):
    # χ² から事後確率 ∝ exp(-χ²/2) をつくり、平均と標準偏差を返す(一様な事前分布、等間隔の格子)
    p = np.exp(-(c2 - np.min(c2)) / 2)
    p /= p.sum()
    mean = np.sum(p * values)
    return mean, np.sqrt(np.sum(p * (values - mean) ** 2))

## 3. 平坦な ΛCDM で $\Omega_m$ を求める

CMB の観測などから、宇宙の空間はほぼ平坦($\Omega_k \approx 0$)と分かっています。
まず平坦を仮定し($\Omega_\Lambda = 1 - \Omega_m$)、パラメータを $\Omega_m$ の 1 つだけにします。

In [ ]:
om_grid = np.linspace(0.01, 1.0, 1000)
c2_flat, a_flat = chi2_models(om_grid, 1 - om_grid)
i_best = np.argmin(c2_flat)
om_best = om_grid[i_best]
om_mean, om_err = posterior_stats(om_grid, c2_flat)
n_dof = len(m_obs) - 2  # Ωm と A

print(f"最良値 Ωm = {om_best:.3f}, χ² = {c2_flat[i_best]:.1f} (自由度 {n_dof})")
print(f"Ωm = {om_mean:.3f} ± {om_err:.3f}  →  ΩΛ = {1 - om_mean:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(om_grid, c2_flat - c2_flat[i_best])
plt.axhline(1, ls="--", c="gray", label="Δχ² = 1 (1σ)")
plt.axvline(om_best, c="tab:red", lw=1)
plt.ylim(0, 30)
plt.xlim(0.1, 0.6)
plt.xlabel("Ωm")
plt.ylabel("Δχ²")
plt.title("flat ΛCDM")
plt.legend()
plt.show()

Brout et al. (2022) の Pantheon+ だけの結果は $\Omega_m = 0.334 \pm 0.018$ です。

宇宙のエネルギーの約 2/3 がダークエネルギー、約 1/3 が物質(そのうち普通の物質は 1/6 ほどで、残りは暗黒物質)ということになります。
$\Omega_m = 1$(物質だけの宇宙)は $\Delta\chi^2$ が数百になり、まったく合いません。

In [ ]:
c2_matter_only = chi2_models(1.0, 0.0)[0][0]
print(f"物質だけの宇宙: Δχ² = {c2_matter_only - c2_flat[i_best]:.0f}")

## 4. 曲率も自由にする: $\Omega_m$ と $\Omega_\Lambda$

平坦の仮定を外して、$\Omega_m$ と $\Omega_\Lambda$ を独立に動かします。
1998 年に Riess et al. と Perlmutter et al. が加速膨張を発見したときに示したのが、この平面の図です。

図には次の線を引きます。

- **平坦な宇宙**: $\Omega_m + \Omega_\Lambda = 1$
- **加速/減速の境目**: 今の減速パラメータ $q_0 = \Omega_m/2 - \Omega_\Lambda$ が 0 になる線。これより上なら今の宇宙は加速している
- **ビッグバンがない領域**: $\Omega_\Lambda$ が大きすぎると、過去にさかのぼっても密度が無限大にならず、収縮から跳ね返った宇宙になる(左上の灰色)

In [ ]:
om_ax = np.linspace(0.0, 1.0, 121)
ol_ax = np.linspace(-0.5, 1.5, 161)
OM, OL = np.meshgrid(om_ax, ol_ax)
c2_curved = chi2_models(OM, OL)[0].reshape(OM.shape)
j = np.unravel_index(np.argmin(c2_curved), OM.shape)
print(f"最良値: Ωm = {OM[j]:.3f}, ΩΛ = {OL[j]:.3f}  (Ωk = {1 - OM[j] - OL[j]:+.3f})")

# 周辺化した値(もう一方のパラメータについて確率を足し合わせる)
p = np.exp(-(c2_curved - c2_curved.min()) / 2)
p_om, p_ol = p.sum(axis=0), p.sum(axis=1)
for name, ax, pp in [("Ωm", om_ax, p_om), ("ΩΛ", ol_ax, p_ol)]:
    pp = pp / pp.sum()
    mu = np.sum(pp * ax)
    print(f"{name} = {mu:.3f} ± {np.sqrt(np.sum(pp * (ax - mu) ** 2)):.3f}")

q0 = OM / 2 - OL
# 「ダークエネルギーなし」「今は減速している」モデルの中で一番よく合うものでも、どれだけ合わないか
d_c2_no_de = np.min(c2_curved[np.isclose(OL, 0)]) - c2_curved.min()
d_c2_decel = np.min(c2_curved[q0 >= 0]) - c2_curved.min()
print(f"ΩΛ = 0 の中で最良でも Δχ² = {d_c2_no_de:.0f}")
print(f"今の宇宙が減速している(q0 ≥ 0)モデルの中で最良でも Δχ² = {d_c2_decel:.0f}")

In [ ]:
# ビッグバンがあるか: z = 0 〜 1000 のどこかで E² ≤ 0 になるなら、ない
z_far = np.logspace(-3, 3, 400)[None, :]
no_big_bang = np.any(e_of_z(z_far, OM.ravel()[:, None], OL.ravel()[:, None]) <= 0, axis=1).reshape(OM.shape)

levels = chi2_dist.ppf([0.683, 0.954, 0.997], df=2)  # 2 パラメータの 1σ, 2σ, 3σ
d_c2 = np.where(np.isfinite(c2_curved), c2_curved - c2_curved.min(), 1e6)

fig, ax = plt.subplots(figsize=(7, 7))
ax.contourf(OM, OL, no_big_bang, levels=[0.5, 1.5], colors=["lightgray"])
ax.contourf(OM, OL, d_c2, levels=[0, *levels], colors=["tab:blue", "cornflowerblue", "lightsteelblue"])
ax.contour(OM, OL, d_c2, levels=levels, colors="tab:blue", linewidths=0.8)
ax.plot(om_ax, 1 - om_ax, "k--", lw=1, label="flat (Ωk = 0)")
ax.plot(om_ax, om_ax / 2, "k:", lw=1, label="q0 = 0 (accelerating above)")
ax.plot(OM[j], OL[j], "w+", ms=10)
ax.plot(1, 0, "rs", label="matter only (Ωm=1)")
ax.text(0.05, 1.35, "no Big Bang", color="gray")
ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel("Ωm")
ax.set_ylabel("ΩΛ")
ax.set_title("Pantheon+ SNe Ia: 1σ, 2σ, 3σ")
ax.legend(loc="lower right")
plt.show()

超新星だけで、$\Omega_\Lambda = 0$(ダークエネルギーなし)は 3σ よりはるかに外です。
等高線は「$\Omega_\Lambda - \Omega_m$ がほぼ一定」の方向に細長く伸びていて、超新星だけでは曲率はあまりよく決まりません。
平坦な宇宙の線とは矛盾なく交わります。

Brout et al. (2022) の結果は $\Omega_m = 0.306 \pm 0.057$, $\Omega_\Lambda = 0.625 \pm 0.084$ です。
CMB や BAO(次のノートブック)は、この等高線とほぼ直交する方向に強い制限を与えるので、組み合わせると $\Omega_k \approx 0$ に絞り込まれます。

## 5. ダークエネルギーの状態方程式 $w$

ダークエネルギーの圧力 $p$ と密度 $\rho$ の比 $w = p / (\rho c^2)$ を状態方程式パラメータといいます。

- $w = -1$: 宇宙項(Λ)。膨張しても密度が変わらない
- $w > -1$: 膨張とともに密度がゆっくり薄まる(クインテッセンスなど)
- $w < -1/3$ でないと、加速膨張を引き起こせない

平坦を仮定し、$\Omega_m$ と $w$(時間によらず一定と仮定)を動かします。

In [ ]:
om_ax_w = np.linspace(0.0, 0.6, 121)
w_ax = np.linspace(-2.0, -0.3, 171)
OMW, W = np.meshgrid(om_ax_w, w_ax)
c2_w = chi2_models(OMW, 1 - OMW, W)[0].reshape(OMW.shape)
jw = np.unravel_index(np.argmin(c2_w), OMW.shape)
print(f"最良値: Ωm = {OMW[jw]:.3f}, w = {W[jw]:.3f}")

pw = np.exp(-(c2_w - c2_w.min()) / 2)
pw_w = pw.sum(axis=1) / pw.sum()
w_mean = np.sum(pw_w * w_ax)
w_err = np.sqrt(np.sum(pw_w * (w_ax - w_mean) ** 2))
print(f"w = {w_mean:.3f} ± {w_err:.3f}  (宇宙項 w = -1 とのずれ {(w_mean + 1) / w_err:.1f}σ)")

d_c2w = np.where(np.isfinite(c2_w), c2_w - c2_w.min(), 1e6)
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(OMW, W, d_c2w, levels=[0, *levels], colors=["tab:green", "mediumseagreen", "honeydew"])
ax.contour(OMW, W, d_c2w, levels=levels, colors="tab:green", linewidths=0.8)
ax.axhline(-1, c="k", ls="--", lw=1, label="cosmological constant (w = -1)")
ax.plot(OMW[jw], W[jw], "k+", ms=10)
ax.set_xlabel("Ωm")
ax.set_ylabel("w")
ax.set_title("flat wCDM: 1σ, 2σ, 3σ")
ax.legend()
plt.show()

Brout et al. (2022) の超新星だけの結果は $w = -0.90 \pm 0.14$ で、宇宙項($w = -1$)と矛盾しません。
ここでも $\Omega_m$ と $w$ は強く相関(縮退)していて、超新星だけでは両方を精度よくは決められません。

> 2024〜2025 年に DESI の BAO と超新星を組み合わせた解析で、「$w$ が時間とともに変わっている」兆候(2.5〜4σ 程度)が報告され、話題になっています。
> $w$ を時間変化させる($w_0 w_a$ モデル)のは発展課題として、次のノートブックで BAO と比べるときに戻ってきます。

## 6. ハッブル定数も同じ当てはめから出す

2 章で見たように、ハッブル図からは $\mathcal{A} = M_B + 5\log_{10}(c/H_0) + 25$ が決まります。
セファイドで距離が分かっている銀河の超新星から $M_B$ を決めれば、$H_0$ が分かります。

前のノートブックと同じ考え方ですが、ここでは較正用の超新星の共分散行列も使って重み付き平均をとります。
$H_0$ を出すのに、$z < 0.15$ の近似式ではなく、3 章で決めた平坦 ΛCDM の距離を全赤方偏移で使っている点が違います。

In [ ]:
cal = (raw["IS_CALIBRATOR"] == 1).to_numpy()
cov_cal_inv = np.linalg.inv(cov_all[np.ix_(cal, cal)])
mb_minus_mu = (raw["m_b_corr"] - raw["CEPH_DIST"]).to_numpy()[cal]
one_cal = np.ones(cal.sum())
M_B = (one_cal @ cov_cal_inv @ mb_minus_mu) / (one_cal @ cov_cal_inv @ one_cal)
M_B_err = 1 / np.sqrt(one_cal @ cov_cal_inv @ one_cal)

A_best = a_flat[i_best]
H0 = C_KMS / 10 ** ((A_best - M_B - 25) / 5)
H0_err = H0 * np.log(10) / 5 * M_B_err  # M_B の誤差から(A の誤差はずっと小さい)
print(f"較正用の超新星 {cal.sum()} 行: M_B = {M_B:.3f} ± {M_B_err:.3f}")
print(f"H0 = {H0:.1f} ± {H0_err:.1f} km/s/Mpc")

前のノートブックとほぼ同じ $H_0 \approx 73$ km/s/Mpc になります。
ただしこの誤差には、セファイドの距離そのもの(視差や LMC 距離による較正)の誤差が入っていません(SH0ES の最終的な誤差は ±1.0)。

## 7. 求めたパラメータで宇宙の膨張の歴史をたどる

平坦な ΛCDM の結果($\Omega_m$ と $H_0$)を使って、膨張がどう変わってきたかを計算します。

### 減速パラメータ $q(z)$ と、加速に転じた時期

$q = -\ddot{a}\,a / \dot{a}^2$ は、膨張にブレーキがかかっている($q > 0$)か、アクセルが踏まれている($q < 0$)かを表します。平坦な ΛCDM では

$$
q(z) = \frac{\tfrac12 \Omega_m (1+z)^3 - \Omega_\Lambda}{\Omega_m (1+z)^3 + \Omega_\Lambda}, \qquad
q = 0 \text{ となる } z_t = \left(\frac{2\Omega_\Lambda}{\Omega_m}\right)^{1/3} - 1
$$

昔は物質の重力が勝って減速していて、物質が薄まるにつれてダークエネルギーが勝ち、加速に転じました。
$\Omega_m$ の不確かさを反映させるため、3 章の事後分布から $\Omega_m$ をたくさん引いて幅も描きます。

In [ ]:
def q_of_z(z, om):
    ol = 1 - om
    x = om * (1 + z) ** 3
    return (x / 2 - ol) / (x + ol)


def z_transition(om):
    return (2 * (1 - om) / om) ** (1 / 3) - 1


def lookback_gyr(z, om, h0):
    # 平坦 ΛCDM のルックバック時間 [Gyr] = 宇宙年齢 - その z での年齢
    return age_gyr(0, om, h0) - age_gyr(z, om, h0)


def age_gyr(z, om, h0):
    # 平坦 ΛCDM で、赤方偏移 z のときの宇宙年齢 [Gyr](解析解)
    ol = 1 - om
    return HUBBLE_TIME_GYR / h0 * 2 / (3 * np.sqrt(ol)) * np.arcsinh(np.sqrt(ol / om) * (1 + z) ** -1.5)


rng = np.random.default_rng(0)
p_flat = np.exp(-(c2_flat - c2_flat.min()) / 2)
om_samples = rng.choice(om_grid, size=4000, p=p_flat / p_flat.sum())

zt = z_transition(om_samples)
zt_lo, zt_mid, zt_hi = np.percentile(zt, [16, 50, 84])
t_ago = lookback_gyr(zt, om_samples, H0)
print(f"加速に転じた赤方偏移 z_t = {zt_mid:.2f} (+{zt_hi - zt_mid:.2f} / -{zt_mid - zt_lo:.2f})")
print(f"  それは約 {np.median(t_ago) * 10:.0f} 億年前")
print(f"今の減速パラメータ q0 = {np.median(q_of_z(0, om_samples)):.2f} ± {np.std(q_of_z(0, om_samples)):.2f}")

zz = np.linspace(0, 3, 300)
q_band = np.percentile(q_of_z(zz[None, :], om_samples[:, None]), [16, 50, 84], axis=0)
plt.figure(figsize=(7, 4))
plt.fill_between(zz, q_band[0], q_band[2], alpha=0.3, label="flat ΛCDM fit (1σ)")
plt.plot(zz, q_band[1])
plt.plot(zz, np.full_like(zz, 0.5), "r--", label="matter only (q = 0.5)")
plt.axhline(0, c="k", lw=0.8)
plt.axvline(zt_mid, c="gray", ls=":", label=f"z_t = {zt_mid:.2f}")
plt.xlabel("redshift z")
plt.ylabel("deceleration parameter q(z)")
plt.text(2.0, 0.1, "decelerating", color="gray")
plt.text(0.05, -0.45, "accelerating", color="gray")
plt.legend(loc="lower right")
plt.show()

### 膨張の「速さ」はいつから増えはじめたか

ハッブルパラメータ $H = \dot{a}/a$ は、今も昔も **減り続けています**。加速しているのに減っているのは、$H$ が「割合」だからです。
宇宙の大きさ $a$ そのものが増える速さ $\dot{a}$ を見ると、加速膨張がはっきり分かります。

$$
\dot{a} = a H = \frac{H(z)}{1 + z} = H_0 \frac{E(z)}{1+z}
$$

$\dot{a}$ は、ある 2 つの銀河(宇宙膨張に乗って動いているもの)の間の距離が広がる速さに比例します。

In [ ]:
def e_flat(z, om):
    return np.sqrt(om * (1 + z) ** 3 + (1 - om))


zz = np.linspace(0, 3, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for om, label, style in [(om_best, f"flat ΛCDM (Ωm={om_best:.2f})", "-"), (1.0, "matter only", "--")]:
    axes[0].plot(zz, H0 * e_flat(zz, om), style, label=label)
    axes[1].plot(zz, H0 * e_flat(zz, om) / (1 + zz), style, label=label)
axes[0].set_ylabel("H(z) [km/s/Mpc]")
axes[0].set_title("Hubble parameter H = (da/dt)/a")
axes[1].set_ylabel("H(z)/(1+z) [km/s/Mpc]")
axes[1].set_title("expansion speed da/dt (in units where a0 = 1)")
axes[1].axvline(zt_mid, c="gray", ls=":")
for ax in axes:
    ax.set_xlabel("redshift z")
    ax.legend()
plt.tight_layout()
plt.show()

右の図で、ΛCDM の $\dot{a}$ は $z_t$ で底を打ち、それより最近($z$ が小さい側)では増えています。これが加速膨張です。
物質だけの宇宙では、$\dot{a}$ はずっと減り続けます。

左の $H(z)$ は、超新星以外の方法(宇宙クロノメーターや BAO)でも直接測れます。次のノートブックで、このグラフにほかの方法の測定点を重ねます。

### 宇宙の大きさの時間変化と、宇宙の年齢

宇宙の大きさ(スケール因子)$a$ の時間変化を、今を $t = 0$ としてさかのぼって描きます。平坦な ΛCDM の宇宙年齢は

$$
t_0 = \frac{1}{H_0} \frac{2}{3\sqrt{\Omega_\Lambda}} \sinh^{-1}\!\sqrt{\frac{\Omega_\Lambda}{\Omega_m}}
$$

です。比較のため、物質だけの宇宙($t_0 = \tfrac{2}{3} H_0^{-1}$)と、何もない宇宙($t_0 = H_0^{-1}$、$\dot{a}$ が一定)も描きます。

In [ ]:
a = np.linspace(1e-4, 1.6, 3000)


def time_of_a(a, om, ol):
    # t(a) [Gyr] = ∫ da / (a H(a)) を数値積分(Ωk = 1 - Ωm - ΩΛ)
    ok = 1 - om - ol
    integrand = 1 / np.sqrt(om / a + ok + ol * a ** 2)
    t = np.concatenate([[0], np.cumsum((integrand[1:] + integrand[:-1]) / 2 * np.diff(a))])
    return t * HUBBLE_TIME_GYR / H0


plt.figure(figsize=(7, 5))
for om, ol, label, style in [
    (om_best, 1 - om_best, f"flat ΛCDM (Ωm={om_best:.2f})", "-"),
    (1.0, 0.0, "matter only", "--"),
    (0.0, 0.0, "empty (coasting)", ":"),
]:
    t = time_of_a(a, om, ol)
    t_now = np.interp(1.0, a, t)
    plt.plot(t - t_now, a, style, label=f"{label}: age {t_now:.1f} Gyr")
plt.axhline(1, c="gray", lw=0.5)
plt.axvline(0, c="gray", lw=0.5)
plt.xlabel("time from now [Gyr]")
plt.ylabel("scale factor a (now = 1)")
plt.xlim(-15, 8)
plt.ylim(0, 1.6)
plt.legend()
plt.show()

t0 = age_gyr(0, om_samples, H0)
print(f"宇宙の年齢(平坦 ΛCDM, H0 = {H0:.1f}): {np.median(t0):.2f} ± {np.std(t0):.2f} Gyr(Ωm の誤差のみ)")
print(f"物質だけの宇宙なら: {2 / 3 * HUBBLE_TIME_GYR / H0:.2f} Gyr")

物質だけの宇宙だと、年齢は約 90 億年になります。これは、球状星団の最も古い星の年齢(約 120〜130 億年)より若く、矛盾します。
加速膨張する ΛCDM では、昔の膨張がゆっくりだったぶん年齢が延びて、約 125 億年になります。

なお、Planck の値($H_0 = 67.4$, $\Omega_m = 0.315$)では 138 億年です。$H_0$ が大きいと宇宙年齢は短くなります(ハッブル・テンションは宇宙年齢にも響きます)。

## まとめ

実行すると、だいたい次のようになります。

| 量 | このノートブック | Brout et al. (2022) / SH0ES |
| --- | --- | --- |
| $\Omega_m$(平坦 ΛCDM) | $0.333 \pm 0.018$ | $0.334 \pm 0.018$ |
| $\Omega_m, \Omega_\Lambda$(曲率あり) | $0.297 \pm 0.054$, $0.610 \pm 0.081$ | $0.306 \pm 0.057$, $0.625 \pm 0.084$ |
| $w$(平坦 wCDM) | $-0.91 \pm 0.15$ | $-0.90 \pm 0.14$ |
| $H_0$ [km/s/Mpc] | $73.4 \pm 1.0$ | $73.0 \pm 1.0$(Riess et al. 2022) |
| 加速に転じた $z_t$ | $0.59 \pm 0.04$(約 53 億年前) | |
| 宇宙の年齢 | 約 125 億年($H_0 = 73.4$ のとき) | 138 億年(Planck) |

## 発展メモ

- 共分散行列を対角成分だけにすると(系統誤差の相関を無視すると)、$\Omega_m$ や $w$ の誤差はどう変わるか
- $z$ の下限を 0.01 から 0.023 にすると結果は変わるか(近い超新星は固有運動の影響を受けやすい)
- $w$ を時間変化させる $w(a) = w_0 + w_a (1 - a)$ に拡張し、$(w_0, w_a)$ 平面で等高線を描く
- 格子ではなく MCMC(`emcee` など)でパラメータを推定してみる

## 参考・クレジット

- Scolnic et al. (2022), *The Pantheon+ Analysis: The Full Data Set and Light-curve Release*, ApJ 938, 113
- Brout et al. (2022), *The Pantheon+ Analysis: Cosmological Constraints*, ApJ 938, 110
- Riess et al. (2022), *A Comprehensive Measurement of the Local Value of the Hubble Constant with 1 km/s/Mpc Uncertainty from the Hubble Space Telescope and the SH0ES Team*, ApJL 934, L7
- Riess et al. (1998), *Observational Evidence from Supernovae for an Accelerating Universe and a Cosmological Constant*, AJ 116, 1009
- Perlmutter et al. (1999), *Measurements of Ω and Λ from 42 High-Redshift Supernovae*, ApJ 517, 565
- Hogg (1999), *Distance measures in cosmology*, arXiv:astro-ph/9905116(距離の式)
- Planck Collaboration (2020), *Planck 2018 results. VI. Cosmological parameters*, A&A 641, A6